# Hydrologic attributes

Notebook to create the file `CAMELS_DE_1h_hydrologic_attributes.csv`.

columns:
- gauge_id
- q_mean [mm/h]
- runoff_ratio [-]
- flow_period_start
- flow_period_end
- flow_perc_complete [-]
- slope_fdc [-]
- hfd_mean [days] (calculated from hourly data, aggregated to daily for this metric)
- Q5 [mm/h]
- Q95 [mm/h]
- high_q_freq [hours/year]
- high_q_dur [hours]
- low_q_freq [hours/year]
- low_q_dur [hours]
- zero_q_freq [-]

In [1]:
from glob import glob
import pandas as pd
import numpy as np
import polars as pl
from tqdm import tqdm

In [2]:
# get camels_ids from hydromet timeseries
ids = [f.split("_")[-1].split(".")[0] for f in glob("../CAMELS-DE-1h/timeseries/CAMELS_DE_1h_hydromet_timeseries_*.csv")]

# sort camels_ids
ids = sorted(ids)

print(f"Total number of stations in CAMELS-DE-1h: {len(ids)}")

Total number of stations in CAMELS-DE-1h: 1611


Note: We filter for complete hydrological years with the same approach as for climatic attributes to ensure consistency.

Hydrologic signatures are calculated at **hourly** resolution to maximize temporal detail. Only `hfd_mean` (half-flow date) is calculated at daily resolution as it measures "days since October 1st".

In [3]:
from datetime import datetime

def filter_complete_hydro_years(df: pl.DataFrame, tolerance: float = 0.05) -> pl.DataFrame:
    """
    Helper function to filter a DataFrame to only include complete hydrological 
    years (October - September). A hydrological year is considered complete if 
    it has less than the specified tolerance of missing values.
    """
    # Ensure time column is datetime
    if "time" not in df.columns:
        raise ValueError("DataFrame must have a 'time' column")
    
    # Parse time column if it's a string (handle timezone if present)
    if df["time"].dtype == pl.Utf8:
        df = df.with_columns([
            pl.col("time").str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S%#z").alias("time")
        ])
    
    # Remove timezone if present (convert to timezone-naive) - BEFORE getting min/max
    if df["time"].dtype.time_zone is not None:
        df = df.with_columns([
            pl.col("time").dt.replace_time_zone(None).alias("time")
        ])
    
    # Cast to microsecond precision to ensure join works
    df = df.with_columns([
        pl.col("time").cast(pl.Datetime("us")).alias("time")
    ])
    
    # Get min and max years
    min_year = df["time"].dt.year().min()
    max_year = df["time"].dt.year().max()
    
    # Create start and end dates for complete hydrological years
    start_date = datetime(min_year - 1, 10, 1, 0, 0, 0)
    end_date = datetime(max_year + 1, 9, 30, 23, 0, 0)
    
    # Create complete hourly date range with microsecond precision
    complete_range = pl.datetime_range(
        start_date, 
        end_date, 
        interval="1h",
        time_unit="us",
        eager=True
    ).alias("time")
    
    # Create DataFrame with complete range and join with existing data
    df_complete = pl.DataFrame({"time": complete_range})
    df = df_complete.join(df, on="time", how="left")
    
    # Calculate hydrological year
    df = df.with_columns([
        pl.when(pl.col("time").dt.month() >= 10)
        .then(pl.col("time").dt.year() + 1)
        .otherwise(pl.col("time").dt.year())
        .alias("hydro_year")
    ])
    
    # Calculate missing value percentage per hydrological year
    missing_per_year = (
        df.group_by("hydro_year")
        .agg([
            pl.col("discharge_vol_obs").is_null().mean().alias("missing_pct")
        ])
    )
    
    # Get years that meet the tolerance threshold
    valid_years = missing_per_year.filter(
        pl.col("missing_pct") <= tolerance
    )["hydro_year"]
    
    # Filter DataFrame to only include valid years
    df_filtered = df.filter(pl.col("hydro_year").is_in(valid_years))
    
    # Drop the hydro_year column
    df_filtered = df_filtered.drop("hydro_year")
    
    return df_filtered

## Function Definitions

Functions to calculate each hydrologic attribute from the processed dataframe.

**Note**: Most hydrologic signatures are calculated at hourly resolution. Only `hfd_mean` requires daily aggregation as it measures seasonal timing in days.

In [6]:
def aggregate_to_daily(df: pl.DataFrame) -> pl.DataFrame:
    """
    Aggregate hourly data to daily for hydrologic signatures.
    Converts discharge from volumetric (m³/s) to specific (mm/d) by summing hourly values.
    """
    # Add date column
    df = df.with_columns([
        pl.col("time").dt.date().alias("date")
    ])
    
    # Aggregate to daily
    df_daily = df.group_by("date").agg([
        # discharge_spec_obs: sum hourly mm/h to get mm/d
        pl.col("discharge_spec_obs").sum().alias("discharge_spec_obs"),
        # discharge_vol_obs: mean of hourly m³/s
        pl.col("discharge_vol_obs").mean().alias("discharge_vol_obs"),
        # precipitation: sum hourly mm/h to get mm/d
        pl.col("precipitation_mean").sum().alias("precipitation_mean")
    ]).sort("date")
    
    # Convert date back to datetime for consistency
    df_daily = df_daily.with_columns([
        pl.col("date").cast(pl.Datetime("us")).alias("time")
    ]).drop("date")
    
    return df_daily


def calculate_q_mean_and_runoff_ratio(df: pl.DataFrame) -> tuple:
    """
    Calculate mean hourly discharge [mm/h] and runoff ratio [-]
    """
    if df.height == 0:
        return np.nan, np.nan
    
    # Calculate means from hourly data
    q_mean = df["discharge_spec_obs"].mean()
    p_mean = df["precipitation_mean"].mean()
    
    if q_mean is None or p_mean is None or p_mean == 0:
        return np.nan, np.nan
    
    runoff_ratio = q_mean / p_mean
    
    return round(q_mean, 2), round(runoff_ratio, 2)


def calculate_flow_period_metrics(df_original: pl.DataFrame) -> tuple:
    """
    Calculate flow_period_start, flow_period_end, and flow_perc_complete
    Note: Does NOT filter for complete hydrological years (uses original data)
    """
    if df_original.height == 0:
        return np.nan, np.nan, np.nan
    
    # Filter to non-null discharge observations
    df_valid = df_original.filter(pl.col("discharge_vol_obs").is_not_null())
    
    if df_valid.height == 0:
        return np.nan, np.nan, np.nan
    
    # flow_period_start
    flow_period_start = df_valid["time"].min()
    
    # flow_period_end
    flow_period_end = df_valid["time"].max()
    
    # flow_perc_complete
    flow_perc_complete = df_valid.height / df_original.height * 100
    
    return flow_period_start, flow_period_end, round(flow_perc_complete, 2)


def calculate_slope_fdc(df: pl.DataFrame) -> float:
    """
    Calculate slope of flow duration curve (between log-transformed 33rd and 66th streamflow percentiles).
    """
    if df.height == 0:
        return np.nan
    
    # Filter out NaN and zero values from hourly data
    df_fdc = df.filter(
        (pl.col("discharge_spec_obs").is_not_null()) & 
        (pl.col("discharge_spec_obs") > 0)
    )
    
    if df_fdc.height == 0:
        return np.nan
    
    # Calculate percentiles
    p33 = df_fdc["discharge_spec_obs"].quantile(0.33)
    p66 = df_fdc["discharge_spec_obs"].quantile(0.66)
    
    if p33 is None or p66 is None or p33 <= 0 or p66 <= 0:
        return np.nan
    
    # Calculate slope
    slope = (np.log(p66) - np.log(p33)) / (0.66 - 0.33)
    
    return round(slope, 2)


def calculate_hfd_mean(df: pl.DataFrame) -> float:
    """
    Calculate mean half-flow date (days since October 1st when cumulative discharge reaches 
    half of annual discharge)
    """
    if df.height == 0:
        return np.nan
    
    # Aggregate to daily
    df_daily = aggregate_to_daily(df)
    
    # Add water year column
    df_daily = df_daily.with_columns([
        pl.when(pl.col("time").dt.month() >= 10)
        .then(pl.col("time").dt.year() + 1)
        .otherwise(pl.col("time").dt.year())
        .alias("water_year")
    ])
    
    # Calculate annual discharge per water year
    annual_discharge = df_daily.group_by("water_year").agg([
        pl.col("discharge_vol_obs").sum().alias("annual_discharge")
    ])
    
    # Calculate half discharge
    annual_discharge = annual_discharge.with_columns([
        (pl.col("annual_discharge") / 2).alias("half_discharge")
    ])
    
    # Join back to get half_discharge for each row
    df_daily = df_daily.join(annual_discharge, on="water_year", how="left")
    
    # Calculate cumulative discharge within each water year
    df_daily = df_daily.sort(["water_year", "time"]).with_columns([
        pl.col("discharge_vol_obs").cum_sum().over("water_year").alias("cumulative_discharge")
    ])
    
    # Find first date where cumulative exceeds half
    df_daily = df_daily.with_columns([
        (pl.col("cumulative_discharge") > pl.col("half_discharge")).alias("exceeds_half")
    ])
    
    # Get first exceeding date per water year
    half_flow_dates = df_daily.filter(pl.col("exceeds_half")).group_by("water_year").agg([
        pl.col("time").min().alias("half_flow_date")
    ])
    
    if half_flow_dates.height == 0:
        return np.nan
    
    # Calculate days since October 1st
    half_flow_dates = half_flow_dates.with_columns([
        pl.when(pl.col("half_flow_date").dt.month() >= 10)
        .then(pl.datetime(pl.col("half_flow_date").dt.year(), 10, 1))
        .otherwise(pl.datetime(pl.col("half_flow_date").dt.year() - 1, 10, 1))
        .alias("october_1st")
    ])
    
    half_flow_dates = half_flow_dates.with_columns([
        ((pl.col("half_flow_date") - pl.col("october_1st")).dt.total_days() + 1).alias("days_since_oct")
    ])
    
    # Calculate mean
    hfd_mean = half_flow_dates["days_since_oct"].mean()
    
    if hfd_mean is None:
        return np.nan
    
    return round(hfd_mean, 2)


def calculate_q5_q95(df: pl.DataFrame) -> tuple:
    """
    Calculate Q5 and Q95 (5% and 95% flow quantiles in mm/h).
    """
    if df.height == 0:
        return np.nan, np.nan
    
    # Calculate quantiles from hourly data
    q5 = df["discharge_spec_obs"].quantile(0.05)
    q95 = df["discharge_spec_obs"].quantile(0.95)
    
    if q5 is None or q95 is None:
        return np.nan, np.nan
    
    return round(q5, 2), round(q95, 2)


def calculate_high_low_flow_metrics(df: pl.DataFrame) -> tuple:
    """
    Calculate high_q_freq, high_q_dur, low_q_freq, low_q_dur, and zero_q_freq
    Uses hourly data - frequencies in hours/yr, durations in hours.
    """
    if df.height == 0:
        return np.nan, np.nan, np.nan, np.nan, np.nan
    
    # Filter out NaN values first
    df_valid = df.filter(pl.col("discharge_spec_obs").is_not_null())
    
    if df_valid.height == 0:
        return np.nan, np.nan, np.nan, np.nan, np.nan
    
    # Calculate median and mean for thresholds from hourly data
    q_median = df_valid["discharge_spec_obs"].median()
    q_mean = df_valid["discharge_spec_obs"].mean()
    
    if q_median is None or q_mean is None:
        return np.nan, np.nan, np.nan, np.nan, np.nan
    
    # High flow events (> 9 times median)
    high_flow = df_valid["discharge_spec_obs"] > 9 * q_median
    high_flow_list = high_flow.to_list()
    
    if any(high_flow_list):
        # Convert to binary string and calculate durations in hours
        hf_bin = ''.join([str(int(x)) for x in high_flow_list])
        hf_dur_list = [len(s) for s in hf_bin.split('0') if len(s) > 0]
        high_q_dur = np.mean(hf_dur_list) if hf_dur_list else 0
        # Frequency: hours/year (8760 hours in a year)
        high_q_freq = sum(high_flow_list) / len(high_flow_list) * 8760
    else:
        high_q_dur = 0
        high_q_freq = 0
    
    # Low flow events (< 0.2 times mean)
    low_flow = df_valid["discharge_spec_obs"] < 0.2 * q_mean
    low_flow_list = low_flow.to_list()
    
    if any(low_flow_list):
        # Convert to binary string and calculate durations in hours
        lf_bin = ''.join([str(int(x)) for x in low_flow_list])
        lf_dur_list = [len(s) for s in lf_bin.split('0') if len(s) > 0]
        low_q_dur = np.mean(lf_dur_list) if lf_dur_list else 0
        # Frequency: hours/year (8760 hours in a year)
        low_q_freq = sum(low_flow_list) / len(low_flow_list) * 8760
    else:
        low_q_dur = 0
        low_q_freq = 0
    
    # Zero flow frequency (check against valid data)
    zero_flow = df_valid["discharge_spec_obs"] == 0
    zero_q_freq = sum(zero_flow.to_list()) / len(zero_flow.to_list()) if len(zero_flow.to_list()) > 0 else 0
    
    return (
        round(high_q_freq, 2), 
        round(high_q_dur, 2), 
        round(low_q_freq, 2), 
        round(low_q_dur, 2), 
        round(zero_q_freq, 2)
    )

## Main Processing Loop

Loop through all station IDs once, reading data once per station and calculating all attributes.

In [7]:
# Reset results dataframe
df_results = pd.DataFrame()

# Main processing loop - read data once per station
for id in tqdm(ids):
    try:
        # Read camels de hydromet timeseries data
        df = pl.read_csv(
            f"../CAMELS-DE-1h/timeseries/CAMELS_DE_1h_hydromet_timeseries_{id}.csv",
            try_parse_dates=True
        )

        # Store original dataframe for flow period metrics (not filtered for complete years)
        df_original = df.clone()

        # Remove timezone from time column BEFORE other operations
        if df["time"].dtype.time_zone is not None:
            df = df.with_columns([
                pl.col("time").dt.replace_time_zone(None).alias("time")
            ])
            df_original = df_original.with_columns([
                pl.col("time").dt.replace_time_zone(None).alias("time")
            ])

        # Make all columns except 'time' float64
        df = df.with_columns([
            pl.col(col).cast(pl.Float64) for col in df.columns if col != "time"
        ])
        df_original = df_original.with_columns([
            pl.col(col).cast(pl.Float64) for col in df_original.columns if col != "time"
        ])
        
        # Filter complete hydrological years
        df = filter_complete_hydro_years(df)

        if df.height == 0:
            print(f"No complete hydrological years for station {id}. Skipping.")
            # Set all attributes to NaN
            df_results.loc[id, "q_mean"] = np.nan
            df_results.loc[id, "runoff_ratio"] = np.nan
            df_results.loc[id, "flow_period_start"] = np.nan
            df_results.loc[id, "flow_period_end"] = np.nan
            df_results.loc[id, "flow_perc_complete"] = np.nan
            df_results.loc[id, "slope_fdc"] = np.nan
            df_results.loc[id, "hfd_mean"] = np.nan
            df_results.loc[id, "Q5"] = np.nan
            df_results.loc[id, "Q95"] = np.nan
            df_results.loc[id, "high_q_freq"] = np.nan
            df_results.loc[id, "high_q_dur"] = np.nan
            df_results.loc[id, "low_q_freq"] = np.nan
            df_results.loc[id, "low_q_dur"] = np.nan
            df_results.loc[id, "zero_q_freq"] = np.nan
            continue

        # Calculate all attributes using the functions
        q_mean, runoff_ratio = calculate_q_mean_and_runoff_ratio(df)
        df_results.loc[id, "q_mean"] = q_mean
        df_results.loc[id, "runoff_ratio"] = runoff_ratio
        
        # Flow period metrics (use original unfiltered data)
        flow_start, flow_end, flow_complete = calculate_flow_period_metrics(df_original)
        df_results.loc[id, "flow_period_start"] = flow_start
        df_results.loc[id, "flow_period_end"] = flow_end
        df_results.loc[id, "flow_perc_complete"] = flow_complete
        
        df_results.loc[id, "slope_fdc"] = calculate_slope_fdc(df)
        df_results.loc[id, "hfd_mean"] = calculate_hfd_mean(df)
        
        q5, q95 = calculate_q5_q95(df)
        df_results.loc[id, "Q5"] = q5
        df_results.loc[id, "Q95"] = q95
        
        high_freq, high_dur, low_freq, low_dur, zero_freq = calculate_high_low_flow_metrics(df)
        df_results.loc[id, "high_q_freq"] = high_freq
        df_results.loc[id, "high_q_dur"] = high_dur
        df_results.loc[id, "low_q_freq"] = low_freq
        df_results.loc[id, "low_q_dur"] = low_dur
        df_results.loc[id, "zero_q_freq"] = zero_freq
        
    except Exception as e:
        print(f"Error processing station {id}: {e}")
        import traceback
        traceback.print_exc()
        # Set all attributes to NaN on error
        df_results.loc[id, "q_mean"] = np.nan
        df_results.loc[id, "runoff_ratio"] = np.nan
        df_results.loc[id, "flow_period_start"] = np.nan
        df_results.loc[id, "flow_period_end"] = np.nan
        df_results.loc[id, "flow_perc_complete"] = np.nan
        df_results.loc[id, "slope_fdc"] = np.nan
        df_results.loc[id, "hfd_mean"] = np.nan
        df_results.loc[id, "Q5"] = np.nan
        df_results.loc[id, "Q95"] = np.nan
        df_results.loc[id, "high_q_freq"] = np.nan
        df_results.loc[id, "high_q_dur"] = np.nan
        df_results.loc[id, "low_q_freq"] = np.nan
        df_results.loc[id, "low_q_dur"] = np.nan
        df_results.loc[id, "zero_q_freq"] = np.nan

# Display results
df_results.head()

  0%|          | 0/1611 [00:00<?, ?it/s]

100%|██████████| 1611/1611 [07:29<00:00,  3.58it/s]


,q_mean,runoff_ratio,flow_period_start,flow_period_end,flow_perc_complete,slope_fdc,hfd_mean,Q5,Q95,high_q_freq,high_q_dur,low_q_freq,low_q_dur,zero_q_freq
DE110000,0.05,0.54,2001-01-01 01:00:00,2024-12-31 23:00:00,100.00,2.10,145.22,0.01,0.17,99.06,36.77,571.29,37.36,0.00
DE110010,0.02,0.17,2001-01-01 01:00:00,2024-12-31 23:00:00,96.51,2.26,146.61,0.00,0.07,912.26,104.03,4182.00,592.25,0.44
DE110020,0.03,0.35,2001-01-01 01:00:00,2024-12-31 23:00:00,100.00,2.20,154.61,0.01,0.10,63.35,48.60,44.53,8.47,0.00
DE110030,0.03,0.36,2001-01-01 01:00:00,2024-12-31 23:00:00,100.00,1.82,160.52,0.01,0.09,4.82,22.20,1.30,2.50,0.00
DE110040,0.04,0.42,2001-01-01 01:00:00,2024-12-31 23:00:00,100.00,0.88,178.91,0.02,0.10,18.03,18.04,0.22,1.25,0.00


## Save results

In [9]:
# save results
df_results.to_csv("../CAMELS-DE-1h/CAMELS_DE_1h_hydrologic_attributes.csv", index_label="gauge_id")